# 6.4.3 Error Correction

:::{admonition} Prompts
:class: tip

- What is the no-cloning theorem? Why does it prevent classical repetition codes but not quantum error correction?
- Walk through the 3-qubit bit-flip code: how does syndrome measurement identify the error without collapsing the logical qubit?
- What is a stabilizer code? How do the stabilizer generators (parity checks) define the code space and enable error detection?
- How does the toric code (§2.3.3) implement quantum error correction topologically? What are the anyonic error syndromes?
- What does the threshold theorem guarantee, and why does it mean quantum computation is scalable in principle?
:::

## Lecture Notes

Decoherence destroys quantum information, but quantum mechanics provides the tools to fight back. Encoding a logical qubit in entangled physical qubits enables error detection and correction without measuring the logical state—a distinction with no classical analog.

### Overview

The no-cloning theorem forbids backup copies, but entanglement enables redundant encoding. Syndromes—measured via stabilizer operators—identify errors without disturbing the logical state. Stabilizer codes and topological codes (toric code, §2.3.3) are robust. The threshold theorem guarantees fault-tolerant computation below a critical error rate.

:::{admonition} Discussion
:class: dropdown tip

The threshold theorem says fault-tolerant quantum computation is possible if the physical error rate $p < p_\text{th}$. For the surface code, $p_\text{th} \approx 1\%$. How does the number of physical qubits per logical qubit scale with $p/p_\text{th}$? What happens as $p \to 0$?
:::

### No-Cloning Theorem

:::{admonition} No-Cloning Theorem
:class: important

There is no universal unitary operator that can copy an arbitrary unknown quantum state:

$$
U(\vert\psi\rangle \otimes \vert 0\rangle) = \vert\psi\rangle \otimes \vert\psi\rangle
$$

for all $\vert\psi\rangle$. This is a direct consequence of the linearity of quantum mechanics.
:::

:::{admonition} Proof: No-Cloning Theorem
:class: dropdown information

Suppose such a unitary $U$ exists. Test on orthogonal states:

$$
U(\vert 0\rangle \otimes \vert 0\rangle) = \vert 0\rangle \otimes \vert 0\rangle, \quad U(\vert 1\rangle \otimes \vert 0\rangle) = \vert 1\rangle \otimes \vert 1\rangle
$$

By linearity, acting on $\vert+\rangle = \frac{1}{\sqrt{2}}(\vert 0\rangle + \vert 1\rangle)$:

$$
U(\vert+\rangle \otimes \vert 0\rangle) = \frac{1}{\sqrt{2}}(\vert 0\rangle \otimes \vert 0\rangle + \vert 1\rangle \otimes \vert 1\rangle)
$$

But cloning would require:

$$
U(\vert+\rangle \otimes \vert 0\rangle) \stackrel{?}{=} \vert+\rangle \otimes \vert+\rangle = \frac{1}{2}(\vert 00\rangle + \vert 01\rangle + \vert 10\rangle + \vert 11\rangle)
$$

These are not equal (different entanglement structure). **Contradiction** — no such $U$ exists.
:::

**Consequence for error correction**: Quantum error correction distributes information across multiple physical qubits without concentrating it in a copyable form.

### Quantum Error Correction: 3-Qubit Bit-Flip Code

**Goal**: Protect one logical qubit from single bit-flip errors.

:::{admonition} 3-Qubit Bit-Flip Code
:class: important

Encode as:

$$
\vert 0\rangle_L = \vert 000\rangle, \quad \vert 1\rangle_L = \vert 111\rangle
$$ (eq-qec-3qubit-encoding)

A general logical state encodes as $\vert\psi\rangle_L = \alpha\vert 000\rangle + \beta\vert 111\rangle$. This is not a copy of $\vert\psi\rangle$ — the superposition is preserved but distributed.

**Key idea**: Measure *what error occurred*, not *what state the qubit is in*.
:::


**Error Scenarios.** A single bit-flip on qubit $j$ maps the logical state to one of three orthogonal error subspaces, each uniquely identifiable.


**Syndrome Measurement.** Measure parity checks without measuring the logical data:

$$
\hat{P}_1 = \hat{Z}_1 \hat{Z}_2 \quad \text{(parity of qubits 1 and 2)}
$$ (eq-qec-parity-check-1)

$$
\hat{P}_2 = \hat{Z}_2 \hat{Z}_3 \quad \text{(parity of qubits 2 and 3)}
$$

| Error Location | $\hat{P}_1$ | $\hat{P}_2$ | Syndrome $(s_1, s_2)$ |
|---|---|---|---|
| None | $+1$ | $+1$ | $(0,0)$ |
| Qubit 1 | $-1$ | $+1$ | $(1,0)$ |
| Qubit 2 | $-1$ | $-1$ | $(1,1)$ |
| Qubit 3 | $+1$ | $-1$ | $(0,1)$ |

**Recovery**: Apply $\hat{\sigma}^x$ to the identified qubit to restore the logical state.

:::{admonition} Construction: Full 3-Qubit Code Details
:class: dropdown note

The encoding circuit uses two CNOT gates: $\vert\psi\rangle\vert 00\rangle \xrightarrow{\text{CNOT}_{12}} \xrightarrow{\text{CNOT}_{13}} \alpha\vert 000\rangle + \beta\vert 111\rangle$. The two syndrome bits distinguish 3 errors plus no error. The 9-qubit Shor code extends this to protect against both bit-flip and phase-flip errors.
:::

:::{admonition} Poll: Quantum error correction and redundancy
:class: dropdown poll

In the three-qubit bit-flip code, a logical qubit $|0_L\rangle = |000\rangle$ and $|1_L\rangle = |111\rangle$ is encoded across three qubits. If a bit-flip error occurs on one qubit, a syndrome measurement on the three qubits identifies which qubit flipped without measuring the logical information. Why does this work?

(A) Three qubits are enough to store a qubit with no errors.

(B) The code uses redundancy: the same logical state is spread across multiple qubits, so local errors can be identified and corrected without destroying the information.

(C) Syndrome measurements project the state into the code space without revealing the logical qubit value.

(D) Quantum information is protected because measurements cannot disturb entangled states.

:::

### Stabilizer Codes

:::{admonition} Stabilizer Code
:class: important

A stabilizer code is defined by a set of commuting Hermitian operators $\{\hat{S}_1, \hat{S}_2, \ldots, \hat{S}_k\}$ (stabilizers) with eigenvalues $\pm 1$. The **code space** is the common $+1$ eigenspace:

$$
\hat{S}_i\vert\psi\rangle_L = +\vert\psi\rangle_L \quad \forall\, i
$$

**Error detection**: An error $\hat{E}$ takes $\vert\psi\rangle_L$ out of the code space if $\hat{E}\hat{S}_i \neq \hat{S}_iE$ (anticommutes with some stabilizer). Measuring $\hat{S}_i$ after the error yields $-1$ — revealing the syndrome without disturbing the logical information.

**Code parameters**: An $[[n, k, d]]$ code uses $n$ physical qubits to encode $k$ logical qubits with distance $d$ (minimum weight of an undetectable logical error), correcting up to $\lfloor(d-1)/2\rfloor$ errors.
:::

:::{admonition} Knill-Laflamme Conditions
:class: important

A code can correct errors $\{\hat{E}_a\}$ if and only if:

$$
\hat{\Pi} \hat{E}_a^\dagger \hat{E}_b \hat{\Pi} = C_{ab}\,\hat{\Pi}
$$ (eq-qec-knill-laflamme)

for all errors $\hat{E}_a, \hat{E}_b$, where $C_{ab}$ is a constant independent of the code state. This ensures that error information is encoded in the syndrome, not in the logical state.
:::

**Examples**: 3-qubit bit-flip $[[3,1,1]]$; Steane code $[[7,1,3]]$; Surface code $[[n,1,\sqrt{n}]]$ (leading candidate).

### Connection to the Toric Code (§2.3.3)

The toric code is a quantum error correcting code with vertex operators $\hat{A}_v = \prod_{j\in v}\hat{\sigma}^x_j$ and plaquette operators $\hat{B}_p = \prod_{j\in p}\hat{\sigma}^z_j$ as stabilizers.

**Errors are anyons**: A bit-flip error creates anyonic $e$-excitations. The syndrome identifies anyon locations.

**Topological protection**: Logical operators must span the torus, giving code distance $d = L$ and exponential suppression with system size.

**Full circle**: The anyons of Ch2 are the error syndromes of Ch6. Topological quantum computing uses anyon braiding for fault-tolerant logic gates — connecting the course's two great themes: topology and quantum information.

### Threshold Theorem

If $p < p_{\mathrm{th}}$, sufficiently large codes make the logical error rate arbitrarily small. Quantum computation is **scalable in principle**.

:::{admonition} Threshold Theorem
:class: important

For a distance-$d$ code (such as the surface code), the logical error rate scales as

$$
p_{\mathrm{logical}} \sim \left(\frac{p}{p_{\mathrm{th}}}\right)^{(d+1)/2}
$$ (eq-qec-threshold-scaling)

If $p < p_{\mathrm{th}}$, increasing $d$ suppresses the logical error exponentially. If $p > p_{\mathrm{th}}$, increasing $d$ makes the logical qubit *worse*.
:::

For surface code, $p_{\mathrm{th}} \approx 1\%$; current qubits: $p \sim 10^{-3}$ (ratio $\approx 0.1$).

:::{admonition} Derivation: Threshold Scaling Law
:class: dropdown information

The scaling {eq}`eq-qec-threshold-scaling` follows from a leading-order combinatorial argument.

**Step 1 — Minimum uncorrectable error weight.** A code with distance $d$ corrects any pattern of up to $t = \lfloor(d-1)/2\rfloor$ errors. A logical failure requires at least

$$
k_{\min} = \left\lceil\frac{d+1}{2}\right\rceil
$$

simultaneous physical errors. For the surface code, this corresponds to an error chain spanning more than halfway across the lattice.

**Step 2 — Failure probability.** Assuming independent errors with probability $p$, the logical error rate is

$$
p_{\mathrm{logical}} = \sum_{k=k_{\min}}^{n} N_k\, p^k (1-p)^{n-k},
$$

where $N_k$ counts the number of weight-$k$ error patterns that cause a logical failure.

**Step 3 — Small-$p$ approximation.** When $p \ll 1$, the sum is dominated by the lowest-weight term $k = k_{\min} = (d+1)/2$:

$$
p_{\mathrm{logical}} \approx N_{\min}\, p^{(d+1)/2},
$$

where $N_{\min}$ is the number of minimum-weight uncorrectable error paths (shortest chains across the lattice in the surface code).

**Step 4 — Normalize to the threshold.** The threshold $p_{\mathrm{th}}$ is defined as the physical error rate where increasing $d$ no longer suppresses $p_{\mathrm{logical}}$. Absorbing $N_{\min}$ into a normalized form gives the standard scaling law:

$$
p_{\mathrm{logical}} \sim \left(\frac{p}{p_{\mathrm{th}}}\right)^{(d+1)/2}.
$$

This is the standard scaling law for distance-$d$ topological codes. $\checkmark$
:::


### Summary

- **Quantum error correction:** Encode logical qubit into many physical qubits; syndrome measurement identifies error type without destroying encoded information; recovery operation restores original state.
- **Stabilizer formalism:** Code stabilized by group of commuting operators $\mathcal{S}$; codewords obey $S\vert\psi_L\rangle = \vert\psi_L\rangle$ for all $S \in \mathcal{S}$. Errors mapped to detectable syndrome eigenvalues.
- **Threshold:** If error rate $p < p_c$ (critical threshold, typically 10^{-2} – 10^{-3}), increasing code distance exponentially improves fidelity (fault-tolerant quantum computing becomes possible).
- **Surface codes:** 2D stabilizer codes with local interactions; scalable and experimentally feasible. Topological protection: errors must form closed loops to cause logical failure.
- **CSS codes:** Concatenate two classical linear codes to protect against bit and phase flips independently; Shor code (9 qubits) and Steane code (7 qubits) are early examples.


## Homework

**1. Three-qubit bit-flip.** Three-Qubit Bit-Flip Code: Encoding and Basis

The 3-qubit bit-flip code encodes logical qubits as:

$$
\vert 0\rangle_L = \vert 000\rangle, \quad \vert 1\rangle_L = \vert 111\rangle
$$


(a) Write the general logical state $\vert\psi\rangle_L = \alpha\vert 0\rangle_L + \beta\vert 1\rangle_L$ explicitly in terms of the physical qubits. Verify that this state lies in the 2-dimensional code subspace of $\mathbb{C}^8$.


(b) Show that a single bit-flip error $\hat{\sigma}^x_j$ on qubit $j$ maps both $\vert 0\rangle_L$ and $\vert 1\rangle_L$ to orthogonal states outside the code subspace.


(c) Explain why the syndrome measurement (parity checks $\hat{P}_1 = \hat{Z}_1 \hat{Z}_2$ and $\hat{P}_2 = \hat{Z}_2 \hat{Z}_3$) does *not* collapse the logical state but *does* reveal which qubit flipped.

**2. Syndrome measurement.** Syndrome Measurement and Error Diagnosis

Consider the 3-qubit bit-flip code. Suppose the error channel acts on qubit 2, applying $\hat{\sigma}^x_2$ to the encoded state $\vert\psi\rangle_L = \alpha\vert 000\rangle + \beta\vert 111\rangle$.


(a) Write the corrupted state after the error occurs.


(b) Calculate the eigenvalues of the corrupted state under the two parity operators $\hat{P}_1 = \hat{Z}_1 \hat{Z}_2$ and $\hat{P}_2 = \hat{Z}_2 \hat{Z}_3$. Verify that the syndrome is $(s_1, s_2) = (1,1)$.


(c) Based on the syndrome table in the lecture notes, which qubit should be corrected? Apply the correction and verify that the original logical state is recovered.

**3. Three-qubit phase-flip.** Three-Qubit Phase-Flip Code

Now consider protection against phase-flip errors. Define the 3-qubit phase-flip code as:

$$
\vert 0\rangle_L = \vert+++\rangle = \frac{1}{2\sqrt{2}}(\vert 000\rangle + \vert 001\rangle + \vert 010\rangle + \vert 011\rangle + \vert 100\rangle + \vert 101\rangle + \vert 110\rangle + \vert 111\rangle)
$$

$$
\vert 1\rangle_L = \vert---\rangle = \frac{1}{2\sqrt{2}}(\vert 000\rangle - \vert 001\rangle - \vert 010\rangle + \vert 011\rangle - \vert 100\rangle + \vert 101\rangle + \vert 110\rangle - \vert 111\rangle)
$$


(a) Show that $\vert+\rangle_L$ and $\vert-\rangle_L$ are eigenstates of $\hat{Z}_1 \hat{Z}_2$ and $\hat{Z}_2 \hat{Z}_3$ with eigenvalue $+1$ and $-1$, respectively.


(b) Explain why a phase-flip error $\hat{\sigma}^z$ on any single qubit can be detected via the same parity checks.


(c) The phase-flip code can be understood as the bit-flip code applied in the Hadamard basis. How does the relationship $\hat{H} \hat{\sigma}^x \hat{H} = \hat{\sigma}^z$ connect the two codes?

**4. Shor Code Construction.** Shor Code: Combining Bit-Flip and Phase-Flip Protection

The 9-qubit Shor code encodes one logical qubit into 9 physical qubits, protecting against both bit-flip and phase-flip errors on any single qubit. The encoding is:

$$
\vert 0\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle + \vert 111\rangle) \otimes (\vert 000\rangle + \vert 111\rangle) \otimes (\vert 000\rangle + \vert 111\rangle)
$$

$$
\vert 1\rangle_L = \frac{1}{2\sqrt{2}}(\vert 000\rangle - \vert 111\rangle) \otimes (\vert 000\rangle - \vert 111\rangle) \otimes (\vert 000\rangle - \vert 111\rangle)
$$

Here each $(\vert 000\rangle + \vert 111\rangle)$ is a 3-qubit phase-flip protected block, and three such blocks protect against bit-flip errors.


(a) How many independent stabilizer generators (parity checks) are required to uniquely identify errors for the Shor code? Explain your reasoning.


(b) Write down *two* of the stabilizer generators explicitly, and explain what error type(s) each detects.


(c) Explain briefly how the Shor code achieves protection against both bit-flip and phase-flip errors with only 9 qubits. Why is this better than naively using two separate 3-qubit codes (which would require 6 qubits)?

**5. Error correction.** Knill-Laflamme Conditions and Code Characterization

The **Knill-Laflamme conditions** state that a quantum error correction code can correct errors in set $\mathcal{E}$ if and only if:

$$
\langle \psi \vert \hat{E}_i^\dagger \hat{E}_j \vert \phi \rangle = \delta_{ij} f(i,j)
$$

for all $\vert\psi\rangle, \vert\phi\rangle$ in the code subspace and all error operators $\hat{E}_i, \hat{E}_j \in \mathcal{E}$, where $f(i,j)$ is independent of $\vert\psi\rangle$ and $\vert\phi\rangle$.


(a) For the 3-qubit bit-flip code, verify the Knill-Laflamme conditions for errors $E \in \{I, \hat{\sigma}^x_1, \hat{\sigma}^x_2, \hat{\sigma}^x_3\}$.


(b) Explain in physical terms why $\hat{E}_i^\dagger \hat{E}_j$ must satisfy this orthogonality structure for error correction to work.


(c) Why do the Knill-Laflamme conditions *not* allow the 3-qubit bit-flip code to correct two or more simultaneous bit-flip errors?

**6. No-Cloning and Error Correction.** No-Cloning and Why Quantum Error Correction Is Non-Trivial

The no-cloning theorem states that an unknown quantum state cannot be universally copied. This creates a fundamental tension with error correction: how can you protect quantum information if you cannot make backup copies?


(a) Explain how the 3-qubit bit-flip code *avoids* cloning by distributing information across three physical qubits *without* copying the original state.


(b) Suppose you attempted to store a backup copy of a logical qubit by trying to create a second independent copy of the entire encoded state. Show that this would violate the no-cloning theorem.


(c) Quantum error correction requires measuring the error syndrome without measuring the logical data. Use the no-cloning theorem to argue why this is *necessary*: if you could measure logical data without destroying it, what would that imply about copying quantum states?

**7. Threshold theorem.** Threshold Theorem and Fault Tolerance (Conceptual)

The **threshold theorem** states: *If the physical error rate per qubit is below a critical threshold $p_{\text{th}}$, then quantum error correction codes can be concatenated recursively, with the logical error rate decreasing exponentially in the number of concatenation layers, enabling arbitrarily large quantum computations.*


(a) In simple terms, explain what "concatenation" of codes means and why it helps reduce errors.


(b) Current superconducting qubits have single-qubit gate errors around $10^{-3}$ to $10^{-4}$. Estimate what threshold $p_{\text{th}}$ would need to be achieved for practical fault-tolerant quantum computers. (Current estimates are $p_{\text{th}} \sim 10^{-3}$ to $10^{-4}$.)


(c) The threshold theorem shows that quantum computing can be **scalable in principle**. What are the remaining practical challenges—beyond just meeting the threshold—that would still stand in the way of a useful large-scale quantum computer?


(d) Surface codes are a leading candidate for fault-tolerant quantum computing. Without working through detailed calculations, explain why a 2D array of qubits (as in the surface code) might be more practical than fully connected qubits for building a large quantum computer, especially regarding error correction overhead.